<a href="https://colab.research.google.com/github/peperjet/algorithm/blob/main/hashtable/hashtable_260428.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

백준 17219 : 비밀번호 찾기

사이트->비밀번호를 저장해놓고 바로 꺼내는 문제

해시테이블(dict 사용)

In [18]:
import sys
from io import StringIO

input_data = """3 2
naver.com abc123
google.com zzz999
daum.net hello
naver.com
daum.net
"""

sys.stdin = StringIO(input_data)
input = sys.stdin.readline

In [19]:
N, M = map(int, input().split())  # N = 저장할 개수, M = 물어볼 개수

# 1단계 : 서랍장(딕셔너리)
password_dict = {}

# 2단계 : N개의 사이트 - 비밀번호를 서랍에 넣기
for _ in range(N):
  site, pw = input().split()
  password_dict[site] = pw # 서랍[사이트이름] = 비밀번호
  # 예) password_dict["naver.com"] = "abc1234"


# 3단계 : M번 물어보는 것에 답하기
for _ in range(M):
  site = input().strip()
  print(password_dict[site]) # 서랍에 꺼내서 출력

abc123
hello


- site = key
- password = value
- pw[site] -> 바로찾기 (빠름)


In [24]:
import builtins
input = builtins.input

password_dict = {
    "naver.com": "abc123",
    "daum.net": "hello",
    "google.com": "zzz999"
}

site = input("사이트 주소 입력: ")
print(password_dict.get(site, "없는 사이트"))

사이트 주소 입력: google.com
zzz999


In [32]:
class HashTable:
  def __init__(self, size=10):
    self.size = size
    # 서랍장 만들기 : 각 서랍은 빈 리스트
    self.table = [[] for _ in range(size)]

  def hash_function(self, key):
    # key를 숫자로 변환하여 서랍 번호 결정
    return sum(ord(c) for c in key) % self.size


  def insert(self, key, value):
    index = self.hash_function(key)
    # 이미 같은 key가 있으면 업데이트

    for i, (k, v) in enumerate(self.table[index]):
      if k == key:
        self.table[index][i] = (key, value)
        return

    # 없으면 뒤에 추가 : 충돌나도 이어붙임
    self.table[index].append((key, value))


  def get(self, key):
    index = self.hash_function(key)
    for k, v in self.table[index]:
      if k == key :
        return v

      return None


체이닝 구현 : 충돌나면 리스트로 이어붙이는 방식을 코드로 만들기

In [34]:
# 위 HashTable 클래스 그대로 사용

ht = HashTable()
ht.insert("naver.com", "abc123")
ht.insert("daum.net", "hello99")
ht.insert("google.com", "qwerty")

site = input("사이트 주소 입력: ")
result = ht.get(site)

if result:
    print(result)
else:
    print("없는 사이트")

사이트 주소 입력: daum.net
hello99


개방 주소법(Open Addressing) : 자리 없으면 옆 칸으로 이동하는 방식 구현 = 코드로
실제로 만들기

In [42]:
class HashTableOpen:
  def __init__(self, size=10):
    self.size = size
    # 서랍장 만들기 : None 빈서랍
    self.table = [None] * size


  def hash_function(self, key):
    return sum(ord(c) for c in key) % self.size


  def insert(self, key, value):
    index = self.hash_function(key)

    # 빈 서랍 찾을 때까지 이동
    while self.table[index] is not None:
      if self.table[index][0] == key: # 같은 key면 업데이트
        self.table[index] = (key, value)
        return
      index = (index + 1) % self.size # 다음 칸으로

    self.table[index] = (key, value)


  def get(self, key):
    index = self.hash_function(key)

    while self.table[index] is not None:
      if self.table[index][0] == key:
        return self.table[index][1]

      index = (index + 1) % self.size # 옆 칸 확인

    return None

In [46]:
ht = HashTableOpen()
ht.insert("naver.com", "abc123")
ht.insert("daum.net", "hello99")   # 충돌나면 옆칸으로!
ht.insert("google.com", "qwerty")

print(ht.get("naver.com"))   # abc123
print(ht.get("daum.net"))    # hello99

abc123
hello99


트리 + 스택 조합

In [54]:
def preorder(tree):
  if not tree:
    return [] # 트리가 비어있으면 끝

  res, stack = [], [0] #  res 결과리스트, stack=0번(A)부터 시작

  while stack: # 스택이 빌 때까지 반복
    index = stack.pop() # 스택에서 하나 꺼내기
    res.append(tree[index]) # 꺼낸 노드 결과에 추가


    # 오른쪽을 먼저 넣어야 왼쪽이 먼저 나옴
    index = 2 * index + 2 # 오른쪽 자식 번호
    if index < len(tree) and tree[index] is not None:
      stack.append(index) # 오른쪽 자식 스택에 넣기


    index -=1
    if index < len(tree) and tree[index] is not None:
      stack.append(index)


  return res

In [55]:
# 테스트 코드
tree = ["A", "B", "C", "D", "E", "F", None, "G"]
print(preorder(tree))

['A', 'B', 'D', 'G', 'E', 'C', 'F']
